### Building a master dataframe
This notebook is for compiling all the disparate data for this project together into one master dataframe to use for visualziations.

The data includes:
- seasion: the GBBO season that featured this technical
- episode: the GBBO episode of that featured this technical
- theme: the theme of the GBBO episode that featured this technical
- techncial_id (gbbo_s#_e#): a unique identifier I created for the technical
- technical_name: the name of the technical challenge recipe
- technical_airdate: the date the episode featuring this technical aired in the U.K.
- technical_time: the time allotted to complete the technical
- is_historical_bake (T/F): an indication of whether the bake is historical, obscure, or highly regionally specific. This data field pertains only to bakes that saw a lift in google search trends and were deemed "non-generic".
- search_baseline_avg: the average search share for the technical in the 7 days prior to the relevant episode air date
- search_post_air_avg: the average search share for the technical in the 7 days after to the relevant episode air date
- search_lift_pct: the percentage in lift between the search_baseline_avg and the search_post_air_avg
- search_lift_note: notes about the search lift (e.g., "new_new" when baseline = 0, else blank)
- r_baking_baseline_share: the share of posts on r/baking about the technical challenge in the 30 days prior to the relevant episode airdate
- r_baking_post_air_share: the share of posts on r/baking about the technical challenge in the 21 days after the relevant episode airdate
- r_baking_lift_diff: the diff between r_baking_baseline_share and r_baking_post_air_share
- r_baking_post_count: the raw count of reddit posts in r/baking about the technical in the 21 days after the relevant episode airdate
- r_bakeoff_baseline_share: the share of posts on r/bakeoff about the technical challenge in the 30 days prior to the relevant episode airdate
- r_bakeoff_post_air_share: the share of posts on r/bakeoff about the technical challenge in the 21 days after the relevant episode airdate
- r_bakeoff_lift_diff: the diff between r_bakeoff_baseline_share and r_bakeoff_post_air_share
- r_bakeoff_post_count: the raw count of reddit posts in r/bakeoff about the technical in the 21 days after the relevant episode airdate
- reddit_raw_post_count: the sum of the count of reddit posts in r/baking and r/bakeoff about the technical in the 21 days after the relevant episode airdate

In [13]:
import pandas as pd
import re

In [16]:
#load in all the relevant data

df_gbbo_technical_information = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/complete/TechnicalBakes.csv')

df_gbbo_technical_google_search_trends = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/complete/gbbo-technical-recipe-search-trends-with-lift-over-baseline.csv')

df_r_baking_lift = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/complete/baking_lift.csv')
df_r_baking_raw_counts = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/complete/r_baking_technical_post_counts.csv')

df_r_bakeoff_lift = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/complete/bakeoff_lift.csv')
df_r_bakeoff_raw_counts = pd.read_csv('/Users/rachelvice/Development/Lede/Projects/bakeoff_effect/data/complete/r_bakeoff_technical_post_counts.csv')

In [17]:
df_gbbo_technical_information.head()

,Airdate,Season,Episode,Theme,Signature,Signature Time (min),Technical,Technical Time (min),Showstopper,Showstopper Time (min),Technical ID
0,8/17/2010,1,1,cake,cake,180.0,victoria sandwich,NaN,chocolate celebration cake,NaN,gbbo_s01_e01
1,8/24/2010,1,2,biscuit,personality biscuits,120.0,scones,60.0,"meringue, choux, and macaron petits fours",240.0,gbbo_s01_e02
2,8/31/2010,1,3,bread,signature bread,210.0,cob,150.0,12 sweet and 12 savoury rolls,360.0,gbbo_s01_e03
3,9/7/2010,1,4,pudding,classic pudding,150.0,mini hot lemon soufflés,40.0,"crumble, bread, & suet puddings",300.0,gbbo_s01_e04
4,9/14/2010,1,5,pastry,savoury pie,150.0,cornish pasties,90.0,savory canapés and sweet tartlets,300.0,gbbo_s01_e05


In [18]:
#cleaning df_gbbo_technical_info
columns_to_drop = [
    'Signature',
    'Signature Time (min)',
    'Showstopper',
    'Showstopper Time (min)'
]

df_gbbo_technical_information_clean = df_gbbo_technical_information.drop(columns=columns_to_drop)
df_gbbo_technical_information_clean

,Airdate,Season,Episode,Theme,Technical,Technical Time (min),Technical ID
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05
...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09


In [19]:
df_gbbo_technical_google_search_trends
df_gbbo_technical_google_search_trends = df_gbbo_technical_google_search_trends.drop(columns = 'Unnamed: 0')
df_gbbo_technical_google_search_trends

,Technical,Airdate,Baseline_Avg,Post_Air_Avg,Lift_Pct
0,victoria sandwich,2010-08-17,25.97,53.75,107.0
1,scones,2010-08-24,10.73,46.25,330.9
2,focaccia,2011-08-30,0.00,12.50,inf
3,miniature pork pies,2011-09-13,0.00,12.50,inf
4,treacle tart,2012-08-28,0.00,23.88,inf
5,queen of puddings,2012-09-18,0.00,12.50,inf
6,fondant fancies,2012-10-16,0.00,28.38,inf
7,florentines,2014-08-13,0.00,25.88,inf
8,ginger biscuits,2017-10-31,2.53,12.50,393.4
9,wagon wheels,2018-08-28,0.00,22.12,inf


In [20]:
df_r_baking_raw_counts['reddit_post_date'] = pd.to_datetime(df_r_baking_raw_counts['reddit_post_date'])
df_r_baking_raw_counts['Airdate'] = pd.to_datetime(df_r_baking_raw_counts['Airdate'])

post_air_mask = df_r_baking_raw_counts['reddit_post_date'] >= df_r_baking_raw_counts['Airdate']
df_r_baking_post_air = df_r_baking_raw_counts[post_air_mask]

In [21]:
df_r_baking_post_air_agg = (
    df_r_baking_post_air
    .groupby('Technical', as_index=False)['post_count']
    .sum()
)

In [22]:
df_r_baking_master = df_r_baking_lift.merge(
    df_r_baking_post_air_agg, on='Technical', how='left'
)
df_r_baking_master

,Technical,baseline_share,post_air_share,zero_baseline_flag,lift,lift_diff,post_count
0,angel cake slices,0.000000,0.000000,True,NaN,0.00000,3
1,arlettes,0.000000,0.000000,True,NaN,0.00000,3
2,baguettes,0.000000,0.000000,True,NaN,0.00000,26
3,beignet soufflés,0.000000,0.000000,True,NaN,0.00000,2
4,belgian buns,0.000000,0.000000,True,NaN,0.00000,0
...,...,...,...,...,...,...,...
60,tart aux pommes,0.008929,0.043478,False,4.869565,0.03455,1
61,tennis cake,0.000000,0.000000,True,NaN,0.00000,4
62,the signature chocolate cake with fresh raspbe...,0.000000,0.000000,True,NaN,0.00000,4
63,tiramisu cake,0.000000,0.000000,True,NaN,0.00000,74


In [23]:
df_r_bakeoff_raw_counts['reddit_post_date'] = pd.to_datetime(df_r_bakeoff_raw_counts['reddit_post_date'])
df_r_bakeoff_raw_counts['Airdate'] = pd.to_datetime(df_r_bakeoff_raw_counts['Airdate'])

post_air_mask = df_r_bakeoff_raw_counts['reddit_post_date'] >= df_r_bakeoff_raw_counts['Airdate']
df_r_bakeoff_post_air = df_r_bakeoff_raw_counts[post_air_mask]

In [24]:
df_r_bakeoff_post_air_agg = (
    df_r_bakeoff_post_air
    .groupby('Technical', as_index=False)['post_count']
    .sum()
)

In [25]:
df_r_bakeoff_master = df_r_bakeoff_lift.merge(
    df_r_bakeoff_post_air_agg, on='Technical', how='left'
)
df_r_bakeoff_master

,Technical,baseline_share,post_air_share,lift_diff,post_count
0,jumbles,0.0,0.021739,0.021739,1
1,miniature pineapple upside-down cakes,0.0,0.002933,0.002933,1
2,raspberry éclairs & salted caramel éclairs,0.0,0.002304,0.002304,1
3,vegan sausage rolls,0.0,0.000000,0.000000,1
4,viennese whirls,0.0,0.015000,0.015000,2
5,æbleskiver,0.0,0.000000,0.000000,1
6,caterpiller cake,0.0,0.000000,0.000000,1
7,dampfnudel,0.0,0.011111,0.011111,2
8,lemon meringue pie,0.0,0.000000,0.000000,0
9,malt loaf,0.0,0.005376,0.005376,1


In [26]:
#normalizing technical names across al dfs so I can merge them into one master DataFrame

def normalize_technical(name):
    name = str(name).strip().lower()
    name = re.sub(r'\b\d+\b', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

for df in [df_gbbo_technical_information_clean, df_gbbo_technical_google_search_trends, df_r_baking_master, df_r_bakeoff_master]:
    df['Technical'] = df['Technical'].apply(normalize_technical)

In [27]:
df_master = (
    df_gbbo_technical_information_clean
    .merge(df_gbbo_technical_google_search_trends.add_prefix('search_').rename(columns={'search_Technical': 'Technical'}), on='Technical', how='left')
    .merge(df_r_baking_master.add_prefix('r_baking_').rename(columns={'r_baking_Technical': 'Technical'}), on='Technical', how='left')
    .merge(df_r_bakeoff_master.add_prefix('r_bakeoff_').rename(columns={'r_bakeoff_Technical': 'Technical'}), on='Technical', how='left')
)

len(df_master)

134

In [28]:
df_master

,Airdate,Season,Episode,Theme,Technical,Technical Time (min),Technical ID,search_Airdate,search_Baseline_Avg,search_Post_Air_Avg,...,r_baking_baseline_share,r_baking_post_air_share,r_baking_zero_baseline_flag,r_baking_lift,r_baking_lift_diff,r_baking_post_count,r_bakeoff_baseline_share,r_bakeoff_post_air_share,r_bakeoff_lift_diff,r_bakeoff_post_count
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01,2010-08-17,25.97,53.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02,2010-08-24,10.73,46.25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06,NaN,NaN,NaN,...,0.000000,0.000000,True,NaN,0.00000,7.0,NaN,NaN,NaN,NaN
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.005952,0.005952,1.0
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08,NaN,NaN,NaN,...,0.000000,0.000000,True,NaN,0.00000,1.0,0.0,0.000000,0.000000,1.0
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09,NaN,NaN,NaN,...,0.008929,0.043478,False,4.869565,0.03455,1.0,0.0,0.008696,0.008696,1.0


In [29]:
#making all cols lowercase

df_master.rename(columns=str.lower, inplace=True)
df_master

,airdate,season,episode,theme,technical,technical time (min),technical id,search_airdate,search_baseline_avg,search_post_air_avg,...,r_baking_baseline_share,r_baking_post_air_share,r_baking_zero_baseline_flag,r_baking_lift,r_baking_lift_diff,r_baking_post_count,r_bakeoff_baseline_share,r_bakeoff_post_air_share,r_bakeoff_lift_diff,r_bakeoff_post_count
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01,2010-08-17,25.97,53.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02,2010-08-24,10.73,46.25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06,NaN,NaN,NaN,...,0.000000,0.000000,True,NaN,0.00000,7.0,NaN,NaN,NaN,NaN
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.005952,0.005952,1.0
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08,NaN,NaN,NaN,...,0.000000,0.000000,True,NaN,0.00000,1.0,0.0,0.000000,0.000000,1.0
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09,NaN,NaN,NaN,...,0.008929,0.043478,False,4.869565,0.03455,1.0,0.0,0.008696,0.008696,1.0


In [30]:
#dropping duplicate airdate col 

df_master = df_master.drop(columns = ['search_airdate'])
df_master

,airdate,season,episode,theme,technical,technical time (min),technical id,search_baseline_avg,search_post_air_avg,search_lift_pct,r_baking_baseline_share,r_baking_post_air_share,r_baking_zero_baseline_flag,r_baking_lift,r_baking_lift_diff,r_baking_post_count,r_bakeoff_baseline_share,r_bakeoff_post_air_share,r_bakeoff_lift_diff,r_bakeoff_post_count
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01,25.97,53.75,107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02,10.73,46.25,330.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06,NaN,NaN,NaN,0.000000,0.000000,True,NaN,0.00000,7.0,NaN,NaN,NaN,NaN
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.005952,0.005952,1.0
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08,NaN,NaN,NaN,0.000000,0.000000,True,NaN,0.00000,1.0,0.0,0.000000,0.000000,1.0
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09,NaN,NaN,NaN,0.008929,0.043478,False,4.869565,0.03455,1.0,0.0,0.008696,0.008696,1.0


In [31]:
#renaming columns to match data dict:

df_master = df_master.rename(columns={
    'Airdate' : 'technical_airdate',
    'Technical' : 'technical_name',
    'technical id' : 'technical_id',
})
df_master

,airdate,season,episode,theme,technical,technical time (min),technical_id,search_baseline_avg,search_post_air_avg,search_lift_pct,r_baking_baseline_share,r_baking_post_air_share,r_baking_zero_baseline_flag,r_baking_lift,r_baking_lift_diff,r_baking_post_count,r_bakeoff_baseline_share,r_bakeoff_post_air_share,r_bakeoff_lift_diff,r_bakeoff_post_count
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01,25.97,53.75,107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02,10.73,46.25,330.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06,NaN,NaN,NaN,0.000000,0.000000,True,NaN,0.00000,7.0,NaN,NaN,NaN,NaN
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.005952,0.005952,1.0
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08,NaN,NaN,NaN,0.000000,0.000000,True,NaN,0.00000,1.0,0.0,0.000000,0.000000,1.0
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09,NaN,NaN,NaN,0.008929,0.043478,False,4.869565,0.03455,1.0,0.0,0.008696,0.008696,1.0


In [32]:
sum_post_counts = df_master['r_baking_post_count'] + df_master['r_bakeoff_post_count']
df_master['reddit_raw_post_count'] = sum_post_counts

df_master

,airdate,season,episode,theme,technical,technical time (min),technical_id,search_baseline_avg,search_post_air_avg,search_lift_pct,...,r_baking_post_air_share,r_baking_zero_baseline_flag,r_baking_lift,r_baking_lift_diff,r_baking_post_count,r_bakeoff_baseline_share,r_bakeoff_post_air_share,r_bakeoff_lift_diff,r_bakeoff_post_count,reddit_raw_post_count
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01,25.97,53.75,107.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02,10.73,46.25,330.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06,NaN,NaN,NaN,...,0.000000,True,NaN,0.00000,7.0,NaN,NaN,NaN,NaN,NaN
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,0.005952,0.005952,1.0,NaN
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08,NaN,NaN,NaN,...,0.000000,True,NaN,0.00000,1.0,0.0,0.000000,0.000000,1.0,2.0
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09,NaN,NaN,NaN,...,0.043478,False,4.869565,0.03455,1.0,0.0,0.008696,0.008696,1.0,2.0


In [33]:
#add historical bake columns - these are defined by the bakes that were brought out of obselescence by GBBO (i.e., they had a 0 search baseline that jumped to an inf lift percent)
historical_bakes = [
    'miniature pork pies',
    'treacle tart',
    'queen of puddings',
    'fondant fancies',
    'florentines',
    'wagon wheels',
    'raspberry blancmange with langues du chat biscuits',
    'malt loaf',
    'angel cake slices',
    'danish kransekake cornucopia cake',
    'custard creams',
    'devonshire splits',
    'dauphinoise pithivier',
]

df_master['is_historical_bake'] = df_master['technical'].isin(historical_bakes)



In [34]:
df_master

,airdate,season,episode,theme,technical,technical time (min),technical_id,search_baseline_avg,search_post_air_avg,search_lift_pct,...,r_baking_zero_baseline_flag,r_baking_lift,r_baking_lift_diff,r_baking_post_count,r_bakeoff_baseline_share,r_bakeoff_post_air_share,r_bakeoff_lift_diff,r_bakeoff_post_count,reddit_raw_post_count,is_historical_bake
0,8/17/2010,1,1,cake,victoria sandwich,NaN,gbbo_s01_e01,25.97,53.75,107.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,8/24/2010,1,2,biscuit,scones,60.0,gbbo_s01_e02,10.73,46.25,330.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2,8/31/2010,1,3,bread,cob,150.0,gbbo_s01_e03,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
3,9/7/2010,1,4,pudding,mini hot lemon soufflés,40.0,gbbo_s01_e04,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
4,9/14/2010,1,5,pastry,cornish pasties,90.0,gbbo_s01_e05,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,10/31/2023,14,6,botanical,lemon and thyme drizzle cake,90.0,gbbo_s14_e06,NaN,NaN,NaN,...,True,NaN,0.00000,7.0,NaN,NaN,NaN,NaN,NaN,False
130,11/7/2023,14,7,desserts,orange and ginger treacle puddings,90.0,gbbo_s14_e07,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.005952,0.005952,1.0,NaN,False
131,11/14/2023,14,8,party,caterpiller cake,150.0,gbbo_s14_e08,NaN,NaN,NaN,...,True,NaN,0.00000,1.0,0.0,0.000000,0.000000,1.0,2.0,False
132,11/21/2023,14,9,patisserie,tart aux pommes,150.0,gbbo_s14_e09,NaN,NaN,NaN,...,False,4.869565,0.03455,1.0,0.0,0.008696,0.008696,1.0,2.0,False


In [35]:
df_master.to_csv('bakeoff_project_master_data.csv')